# Practical 5: Feature Engineering for Anomaly Detection
**Course:** BE05000231 - PDS (Semester V, GTU GSET)
**Course Outcome:** CO3

### Objective:
Extract dual-level numeric and categorical features for machine learning: request-level syntactic, statistical, and Shannon entropy indicators, plus IP-level sliding time-window features (1m and 5m).

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')
import pandas as pd

from pipeline.parser import parse_file
from pipeline.cleaner import clean
from pipeline.labeler import load_rules, label_requests
from pipeline.features import add_request_features, add_window_features, generate_feature_catalogue, shannon_entropy

LOG_PATH = Path('../tests/fixtures/sample_access.log')
RULES_PATH = Path('../rules/web_rules.yaml')


## 2. Ingest, Parse, Clean, and Label Base Logs

In [ ]:
raw_df, _ = parse_file(LOG_PATH)
cleaned_df = clean(raw_df)
rules = load_rules(RULES_PATH)
labeled_df = label_requests(cleaned_df, rules)
print(f'Input records: {len(labeled_df)}')


## 3. Shannon Information Entropy Demonstration

In [ ]:
urls = [
    '/index.html',
    '/products.php?id=1',
    '/search?q=shoes',
    '/products.php?id=1%27%20UNION%20SELECT%20null,username,password%20FROM%20users--',
    '/items.php?cat=shoes%2527%20OR%201%3D1--'
]
for u in urls:
    print(f'Entropy: {shannon_entropy(u):.4f} | URL: {u}')


## 4. Engineer Request-Level Features

In [ ]:
req_feat = add_request_features(labeled_df)
request_cols = [
    'url_length', 'query_length', 'special_chars', 'special_ratio',
    'url_entropy', 'digit_ratio', 'has_sql_kw', 'has_traversal',
    'ua_class', 'is_404', 'method_is_post'
]
req_feat[request_cols].head(8)


## 5. Engineer IP-Level Sliding Time-Window Features (5-minute window)

In [ ]:
win_feat = add_window_features(req_feat, window='5min')
window_cols = [
    'ip', 'ts', 'win_req_count', 'win_ratio_404', 'win_unique_paths',
    'win_path_scan_ratio', 'win_gap_mean', 'win_gap_std'
]
win_feat[window_cols].tail(10)


## 6. Feature Comparison Across Attack Categories

In [ ]:
print('Mean URL Length by Label:')
print(win_feat.groupby('label')['url_length'].mean())

print('\nMean URL Entropy by Label:')
print(win_feat.groupby('label')['url_entropy'].mean())

print('\nMean 404 Ratio in Window by Label:')
print(win_feat.groupby('label')['win_ratio_404'].mean())


## 7. Master Feature Catalogue Export

In [ ]:
cat_path = Path('../data/processed/feature_catalogue.csv')
catalogue = generate_feature_catalogue(cat_path)
print(f'Feature Catalogue exported to {cat_path} with {len(catalogue)} features:')
catalogue[['feature', 'level', 'type', 'description']]


### Conclusion:
Practical 5 extracted 20+ features across request and temporal sliding window levels, providing signals that clearly separate scanning, brute force, and injection attacks from normal web browsing.